# Multi-factor screening

Apply Benjamini-Hochberg-Yekutieli (BHY) FDR control to a batch of
candidate factors. Demonstrates the explicit-family contract and
the duplicate-identity defense (#161) — the behaviour that is
impossible to learn from docstrings alone.


## Factor type

This recipe uses `multi_factor.bhy(...)` over a list of
`FactorProfile` objects. Each profile is produced by
`evaluate(panel, cfg)` for any registered cell — screening is
factor-type-agnostic.

The input list **is** the family. Each profile must carry a unique
`identity = (factor_id, forward_periods)` (#160 anti-shopping
defense); set distinct `factor_id` per candidate via
`dataclasses.replace`. Pass `expand_over=[<context key>]` to declare
per-bucket independent step-ups (Benjamini & Bogomolov 2014
selective inference). Mixing `forward_periods` without `expand_over`
emits `RuntimeWarning` — different horizons carry different null
distributions and pooling silently inflates FDR.

Literature: [Benjamini & Yekutieli (2001)](../reference/bibliography.md).


## Use this when

- You have ≥2 candidate factors and want type-I error control under
  multiple testing.
- Candidates are evaluable under the same `AnalysisConfig` (same
  scope, signal, metric, forward horizon). Mixed cells / horizons
  in one call are caller's responsibility — see § 4 below.
- You prefer FDR control (BHY) over family-wise error (Bonferroni)
  — appropriate when discoveries can tolerate a known false-positive
  rate and power matters.

## What it tests

For an input family of size `N`, BHY step-up keeps profiles whose
ranked p-values satisfy `p_(k) ≤ q · k / (N · c(N))` where
`c(N) = Σ 1/i` (Benjamini-Yekutieli adjustment for arbitrary
dependence). Pass your nominal `q` directly — the `c(N)` correction
is baked in. Cross-family aggregation is *not* performed; that is
the user's responsibility and is deliberately out of scope.

## Output to read

1. `len(survivors)` vs `len(profiles)` — coarse hit rate.
2. Per-survivor `profile.primary_p` and `profile.factor_id` — which
   factor cleared the family-adjusted bar.
3. Any emitted `RuntimeWarning` — flags mixed `forward_periods`
   without `expand_over` (FDR-inflation foot-gun) or singleton
   `expand_over` buckets (BHY on n=1 is a raw cutoff).


## 1. Setup


In [1]:
from __future__ import annotations

import factrix as fl
import numpy as np
import polars as pl
from factrix.preprocess import compute_forward_return

print("factrix version:", fl.__version__)

factrix version: 0.10.0


## 2. Build a single-family batch

Five candidate factors, all under the same
`individual_continuous(IC, forward_periods=5)` cell — a *valid*
BHY input where the step-up actually controls FDR.

We start from one ground-truth factor and add increasing IID noise
to produce variants with varying signal strengths.


In [2]:
import dataclasses

raw = fl.datasets.make_cs_panel(
    n_assets=100,
    n_dates=500,
    ic_target=0.08,
    seed=2024,
)
panel = compute_forward_return(raw, forward_periods=5)
cfg = fl.AnalysisConfig.individual_continuous(
    metric=fl.Metric.IC,
    forward_periods=5,
)


def with_noise(base: pl.DataFrame, scale: float, seed: int) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    return base.with_columns(
        pl.Series(
            "factor",
            base["factor"].to_numpy() + scale * rng.standard_normal(base.height),
        ),
    )


candidates = {
    f"variant_{i}": with_noise(panel, scale=0.5 + 0.3 * i, seed=100 + i)
    for i in range(5)
}
profiles = [
    dataclasses.replace(fl.evaluate(p, cfg), factor_id=name)
    for name, p in candidates.items()
]
for prof in profiles:
    print(
        f"  {prof.factor_id:12s} primary_p={prof.primary_p:.4g}  verdict={prof.verdict()}"
    )

  variant_0    primary_p=2.136e-39  verdict=pass
  variant_1    primary_p=4.052e-26  verdict=pass
  variant_2    primary_p=6.682e-22  verdict=pass
  variant_3    primary_p=3.987e-17  verdict=pass
  variant_4    primary_p=7.407e-14  verdict=pass


## 3. Apply BHY

The input list **is** the family. `bhy` runs one Benjamini-Yekutieli
step-up over all profiles, returns survivors in input order, and
controls FDR ≤ `q` under arbitrary dependence (the `c(N)` correction
is applied internally — no manual adjustment).


In [3]:
survivors = fl.multi_factor.bhy(profiles, q=0.05)
print(f"BHY survivors: {len(survivors)} / {len(profiles)}")
for prof in survivors:
    print(f"  {prof.factor_id:12s} primary_p={prof.primary_p:.4g}")

BHY survivors: 5 / 5
  variant_0    primary_p=2.136e-39
  variant_1    primary_p=4.052e-26
  variant_2    primary_p=6.682e-22
  variant_3    primary_p=3.987e-17
  variant_4    primary_p=7.407e-14


## 4. Duplicate-identity defense

`evaluate()` stamps `factor_id="factor"` by default (the source
column name). If you forget to override it before passing the
profiles to `bhy()`, every candidate lands at the same identity
`("factor", forward_periods)` and `bhy` raises `UserInputError`
rather than silently treating distinct candidates as one
hypothesis.

The fix is what we already did in § 2: stamp a unique `factor_id`
per profile via `dataclasses.replace`. Below we deliberately *skip*
that step to surface the error.


In [4]:
cfg_fm = fl.AnalysisConfig.individual_continuous(
    metric=fl.Metric.FM,
    forward_periods=5,
)

# Both profiles default to factor_id="factor" — same identity.
unstamped = [
    fl.evaluate(panel, cfg),  # IC
    fl.evaluate(panel, cfg_fm),  # FM — different cell, same identity
]
try:
    fl.multi_factor.bhy(unstamped, q=0.05)
except fl.UserInputError as exc:
    print("UserInputError raised as expected:")
    print(str(exc))

UserInputError raised as expected:
bhy(): invalid profiles=('factor', 5)
  Expected: unique partition key across input; duplicate first seen at index 0, again at 1. Set distinct factor_id per profile, or pass expand_over=[<context key>] to declare per-bucket families
  Docs: https://awwesomeman.github.io/factrix/api/bhy#partition-key


## 5. Where to go next

For the broader `n_assets` × factory behaviour matrix see
[Guides § PANEL vs TIMESERIES](../guides/panel-timeseries.md);
for the BHY contract specifically see
[Guides § Batch screening](../guides/batch-screening.md).


In [5]:
print("multi_factor_screening: ok")

multi_factor_screening: ok
